# clustering

The optimal number of clusters was determined using the training set via elbow and silhouette analysis. Cluster stability was subsequently evaluated on the test set. Finally, the clustering model was refit on the full dataset to derive the final interpretable user segments.

In [2]:
# basic modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sentence_transformers import SentenceTransformer 

# pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

# cluster
import hdbscan

# metrics
from sklearn.metrics import silhouette_score

# graphics
import seaborn as sns

pd.set_option('display.float_format', '{:.2f}'.format)

from pathlib import Path

# embedding
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import umap





c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# read data

df = pd.read_pickle("C:/Users/heike/Desktop/Stackfuel/Portfolio/llm-sustainability-analysis/01_data/03_features/conversation_features.pkl")


In [4]:
# orthographic error rate <0.58 for cleaner text

df[df["orthographic_error_rate"]>0.57] # n=53
df=df[df["orthographic_error_rate"]<0.58]
df.shape


(52537, 28)

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 57357.05it/s]


In [6]:
texts = df["first_prompt"].fillna("").tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True
)

Batches: 100%|██████████| 821/821 [13:32<00:00,  1.01it/s]  


In [8]:
df["embedding"] = list(embeddings)

In [11]:
features = np.vstack(embeddings)

features_reduced = PCA(n_components=50).fit_transform(features)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=30,
    metric="euclidean",
    core_dist_n_jobs=-1
)


df["cluster"] = clusterer.fit_predict(features_reduced)

In [12]:
# label clusters

cluster_texts = df.groupby("cluster")["first_prompt"].apply(list)

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

cluster_names = {}

for cluster_id, texts in cluster_texts.items():

    if cluster_id == -1:
        continue

    features = vectorizer.fit_transform(texts)

    nmf = NMF(n_components=1, random_state=42)
    nmf.fit(features)

    feature_names = vectorizer.get_feature_names_out()
    topic_words = nmf.components_[0].argsort()[-8:]

    cluster_names[cluster_id] = [
        feature_names[i] for i in topic_words
    ]

c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
c:\Users\heike\D

In [13]:
def make_label(words):
    return " / ".join(words[:3])

df["cluster_label_nmf"] = df["cluster"].map(
    lambda c: make_label(cluster_names.get(c, ["Noise"]))
)

In [14]:
df["cluster_label_nmf"].value_counts()

cluster_label_nmf
Noise                                   50157
ai / test / hello                         711
flush / gets / new                        230
computing / explain / quantum             183
interview / wait / answers                116
write / english / applications             95
date / time / responses                    90
http / javascript / make                   80
clause / need / nsw                        67
pregnancy / voice / write                  67
chatgpt / don / country                    65
way / information / gpt                    64
rules / time / information                 62
1got / old / ideas                         62
follow / users / need                      58
draft / context / assistance               48
linux / pwd / want                         46
output / generate / content                45
include / area / itinerary                 39
use / want / outrank                       39
search / meta / english                    38
graduates / 20 /

In [ ]:
reducer = umap.UMAP()
features_2d = reducer.fit_transform(features)

In [ ]:
df["umap_x"] = features_2d[:, 0]
df["umap_y"] = features_2d[:, 1]

In [ ]:
# plots

fig = px.scatter(
    df,
    x="umap_x",
    y="umap_y",
    color="cluster",
    hover_data=["first_prompt"]
)

fig.show()

# typical examples per cluster by probabilities

In [128]:
df["cluster_prob"] = clusterer.probabilities_

In [129]:
def show_typical_prompts_all_clusters(
    df,
    cluster_col="macro_cluster",
    prompt_col="first_prompt",
    prob_col="cluster_prob",
    n=5
):
    """
    Zeigt die typischsten Prompts pro Cluster
    basierend auf HDBSCAN membership probability.
    """

    clusters = sorted(df[cluster_col].unique())

    for cluster_id in clusters:

        print("\n" + "=" * 80)
        print(f"CLUSTER {cluster_id}")
        print("=" * 80)

        subset = (
            df[df[cluster_col] == cluster_id]
            .sort_values(prob_col, ascending=False)
            .head(n)
        )

        for i, (_, row) in enumerate(subset.iterrows(), start=1):

            print(f"\n--- Prompt {i} ---")
            print(f"Task Type: {row['task_type']}")
            print(f"Target Cost: {row['target_cost']:.2f}")
            print(f"Target Depth: {row['target_depth']:.2f}")
            print(f"Cluster Prob: {row[prob_col]:.4f}")

            print("\nFULL PROMPT:")
            print(row[prompt_col])

            print("\n" + "-" * 80)

In [130]:
show_typical_prompts_all_clusters(df, n=10)


CLUSTER -1

--- Prompt 1 ---
Task Type: general_assistance
Target Cost: 8.44
Target Depth: 3.43
Cluster Prob: 1.0000

FULL PROMPT:
I need a report that includes 4 sections for a couple of papers, the sections include:
section 1: Challenges and Issues
section 2: Methodology
section 3: Applications
section 4: Pros and Cons
Note: Each section should be under 30 words
Can you do it?



--------------------------------------------------------------------------------

--- Prompt 2 ---
Task Type: brainstorming
Target Cost: 7.85
Target Depth: 2.56
Cluster Prob: 1.0000

FULL PROMPT:
I want you to become my Prompt Creator. Your goal is to help me craft the best possible prompt for my needs. The prompt will be used by you, ChatGPT. You will follow the following process: 1. Your first response will be to ask me what the prompt should be about. I will provide my answer, but we will need to improve it through continual iterations by going through the next steps. 2. Based on my input, you will gener

In [ ]:
base_path = Path(
    "C:/Users/heike/Desktop/Stackfuel/Portfolio/llm-sustainability-analysis/01_data/04_clusters/conversation_features_embeddings_clusters.csv"
)

base_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(base_path, index=False)

# Insights

- Clustering = diagnosis of structure
- HDBSCAN performs significantly better than K-Means
- A task-dominated island structure was identified
- A large noise cluster is normal and informative
- Prompt-level structure is secondary to task-level structure